## DATA rensning

In [24]:
import pandas as pd
import os
from functools import reduce

BASE = "/Users/PC/Documents/Speciale/Analyse/Speciale"
RAW_DIR = os.path.join(BASE, "raw")
CLEAN_DIR = os.path.join(BASE, "clean")
os.makedirs(CLEAN_DIR, exist_ok=True)

### Bloomberg data renses

In [25]:
import pandas as pd
from functools import reduce
import os

RAW = os.path.join(RAW_DIR, "bloomberg_indices_monthly.xlsx")

def read_bbg_sheet(fname, sheet):
    """Læs ét Bloomberg-ark: find 'Date'-rækken, behold Date + PX_LAST, normalisér til månedsslut."""
    raw = pd.read_excel(fname, sheet_name=sheet, header=None)
    h = raw.index[raw[0] == "Date"][0]          # find header-rækken dynamisk
    data = raw.iloc[h+1:].copy()
    data.columns = raw.iloc[h].tolist()
    data = data[["Date", "PX_LAST"]].dropna(subset=["Date"])
    data["Date"] = pd.to_datetime(data["Date"])
    data["PX_LAST"] = pd.to_numeric(data["PX_LAST"], errors="coerce")
    data["month"] = data["Date"].dt.to_period("M").dt.to_timestamp("M")  # månedsslut
    return (data[["month", "PX_LAST"]]
            .rename(columns={"PX_LAST": sheet})
            .sort_values("month").reset_index(drop=True))

xl = pd.ExcelFile(RAW)
frames = [read_bbg_sheet(RAW, s) for s in xl.sheet_names]

bbg = reduce(lambda l, r: pd.merge(l, r, on="month", how="outer"), frames)
bbg = bbg.sort_values("month").reset_index(drop=True)
bbg.columns = [c.replace(" Index", "") for c in bbg.columns]   # ryd navne

# afgræns til 2010-01-31 og frem
bbg = bbg[bbg["month"] >= "2010-01-31"].reset_index(drop=True)

# opret clean-mappe og gem
clean_dir = "/Users/PC/Documents/Speciale/Analyse/Speciale/clean"
os.makedirs(clean_dir, exist_ok=True)
out = os.path.join(clean_dir, "bloomberg_monthly.csv")
bbg.to_csv(out, index=False)

print(f"Gemt: {out}")
print("Form:", bbg.shape)
print(bbg.tail())

Gemt: /Users/PC/Documents/Speciale/Analyse/Speciale/clean/bloomberg_monthly.csv
Form: (200, 30)
         month  EURUSD  USDJPY  USDCHF  GBPUSD  USDDKK  EUR3M   JPY3M  CHF3M  \
195 2026-04-30  1.1731  156.59  0.7814  1.3604  6.3703  47.30 -118.85 -76.68   
196 2026-05-31  1.1659  159.27  0.7810  1.3456  6.4090  44.64 -117.72 -76.54   
197 2026-06-30  1.1422  162.55  0.8084  1.3262  6.5441  43.94 -120.36 -81.42   
198 2026-07-31  1.1527  157.40  0.8075  1.3483  6.4861  42.06 -115.14 -81.31   
199 2026-08-31  1.1618  159.74  0.8084  1.3549  6.4341  41.06 -113.25 -82.21   

     GBP3M  ...  SFBS5  BPBS3  BPBS5  DKBS3  DKBS5  USSO3  EUSWEC  JYSO3S  \
195  -4.71  ...    NaN    NaN    NaN    NaN    NaN    NaN     NaN    1.62   
196  -2.98  ...    NaN    NaN    NaN    NaN    NaN    NaN     NaN    1.63   
197   0.18  ...    NaN    NaN    NaN    NaN    NaN    NaN     NaN    1.67   
198   0.46  ...    NaN    NaN    NaN    NaN    NaN    NaN     NaN    1.83   
199   1.75  ...    NaN    NaN    NaN  

### GPR DATA INDLÆSES

In [26]:
# ============================================================
# GPR — geopolitisk risiko (månedlig), Caldara & Iacoviello
# ============================================================

gpr_raw = pd.read_excel(os.path.join(RAW_DIR, "gpr_geopolitical_risk_monthly.xls"))

# behold hovedindeks + threats/acts-underindeks
gpr = gpr_raw[["month", "GPR", "GPRT", "GPRA"]].copy()

# month er allerede datetime (xlrd konverterer) → normalisér til månedsslut
gpr["month"] = pd.to_datetime(gpr["month"]).dt.to_period("M").dt.to_timestamp("M")

# drop tomme (den historiske serie går tilbage til 1900 uden Recent-GPR)
gpr = gpr.dropna(subset=["GPR"])

# afgræns til 2010-01-31 og frem
gpr = gpr[gpr["month"] >= "2010-01-31"].reset_index(drop=True)

gpr.to_csv(os.path.join(CLEAN_DIR, "gpr_monthly.csv"), index=False)
print("GPR:", gpr.shape)
print(gpr.head(2))

GPR: (200, 4)
       month        GPR       GPRT        GPRA
0 2010-01-31  91.581024  84.972992  100.410942
1 2010-02-28  80.725357  78.846275   80.711739


### TPU data 

In [27]:
# TPU — handelspolitisk usikkerhed
tpu = pd.read_excel(os.path.join(RAW_DIR, "TPU_trade_policy_uncertainty_monthly.xlsx"),
                    sheet_name="TPU_MONTHLY")
tpu = tpu[["DATE", "TPU"]].copy()
tpu["month"] = pd.to_datetime(tpu["DATE"]).dt.to_period("M").dt.to_timestamp("M")
tpu = tpu[tpu["month"] >= "2010-01-31"][["month", "TPU"]].reset_index(drop=True)
tpu.to_csv(os.path.join(CLEAN_DIR, "tpu_monthly.csv"), index=False)

### VIX data

In [28]:
# VIX — dagligt -> månedsgennemsnit
vix = pd.read_csv(os.path.join(RAW_DIR, "vix_volatility_fred_daily.csv"))
vix["month"] = pd.to_datetime(vix["observation_date"]).dt.to_period("M").dt.to_timestamp("M")
vix = vix.groupby("month")["VIXCLS"].mean().reset_index()
vix = vix[vix["month"] >= "2010-01-31"].reset_index(drop=True)
vix.to_csv(os.path.join(CLEAN_DIR, "vix_monthly.csv"), index=False)

### Dollar-indeks FRED

In [29]:
# Bred dollar-indeks — dagligt -> månedsgennemsnit (samme mønster som VIX)
usd = pd.read_csv(os.path.join(RAW_DIR, "usd_broad_dollar_index.csv"))
usd["month"] = pd.to_datetime(usd["observation_date"]).dt.to_period("M").dt.to_timestamp("M")
usd = usd.groupby("month")["DTWEXBGS"].mean().reset_index()
usd = usd[usd["month"] >= "2010-01-31"].reset_index(drop=True)
usd.to_csv(os.path.join(CLEAN_DIR, "usd_index_monthly.csv"), index=False)

### CRSP aktier 
Datoer er dd/mm/yyyy

In [30]:
# CRSP aktier — dato er dd/mm/yyyy
eq = pd.read_csv(os.path.join(RAW_DIR, "us_equity_returns_crsp_monthly.csv"))
eq["month"] = pd.to_datetime(eq["mthcaldt"], format="%d/%m/%Y").dt.to_period("M").dt.to_timestamp("M")
eq = eq[eq["month"] >= "2010-01-31"][["month", "vwretd", "sprtrn"]].reset_index(drop=True)
eq.to_csv(os.path.join(CLEAN_DIR, "us_equity_monthly.csv"), index=False)

### Nationalbanken data

In [32]:
# ============================================================
# DNFPVALE — dansk pensionssektors valutaafdækning (Nationalbanken)
# Bred form: to header-rækker (valuta × post). Enhed: mia. kr.
# ============================================================
import re

RAW = os.path.join(RAW_DIR, "Nationalbanken_pension_hedging_monthly.csv")
dnb = pd.read_csv(RAW, sep=";", encoding="latin-1", skiprows=3, header=[0, 1])

ccy_map = {"Alle valutaer ekskl. DKK": "ALL", "EUR": "EUR", "USD": "USD"}
post_map = {
    "1. Valutaeksponering i alt": "eksp_total",
    "1.1. Afdækket valutaeksponering": "afd_total",
    "1.1.1. Afdækket til kroner": "afd_dkk",
    "1.1.2. Afdækket til euro": "afd_eur",
    "1.1.3. Afdækket til dollar": "afd_usd",
    "1.1.4. Afdækket til øvrige valutaer": "afd_ovr",
    "1.2. Uafdækket valutaeksponering": "uafd_total",
}

# byg pæne kolonnenavne: valuta_post. Valuta-niveauet "fylder ud" (merged celler)
new_cols, last_ccy = [], None
for i, (top, bot) in enumerate(dnb.columns):
    top = str(top).strip()
    bot = re.sub(r"\.\d+$", "", str(bot).strip())   # pandas tilføjer .1/.2 til gentagne navne
    if i == 0:
        new_cols.append("month"); continue
    if top and not top.startswith("Unnamed"):
        last_ccy = ccy_map.get(top, top)
    new_cols.append(f"{last_ccy}_{post_map.get(bot, bot)}")
dnb.columns = new_cols

# tid (2015M01) -> månedsslut
dnb["month"] = (pd.to_datetime(dnb["month"].astype(str).str.strip().str.replace("M", "-"),
                               format="%Y-%m")
                  .dt.to_period("M").dt.to_timestamp("M"))
dnb = dnb.sort_values("month").reset_index(drop=True)

# afdækningsgrader = afdækket i alt / eksponering i alt
for c in ["USD", "EUR", "ALL"]:
    dnb[f"{c}_hedge_ratio"] = dnb[f"{c}_afd_total"] / dnb[f"{c}_eksp_total"]

dnb.to_csv(os.path.join(CLEAN_DIR, "dnfpvale_monthly.csv"), index=False)
print("DNFPVALE:", dnb.shape)
print(dnb[["month", "USD_eksp_total", "USD_afd_total", "USD_hedge_ratio"]].tail(3))

DNFPVALE: (139, 25)
         month  USD_eksp_total  USD_afd_total  USD_hedge_ratio
136 2026-05-31         1887.53        1252.97         0.663815
137 2026-06-30         1923.12        1234.94         0.642154
138 2026-07-31         1916.06        1231.30         0.642621
